
# Trabajo Práctico Aprendizaje Automático 2

### Configuración Automática del Espacio de Trabajo (Estructura `src/`)

------------------------------>Correr esto únicamente si usamos la opción A que está en el README. <-------------------

In [1]:
import os

# Crear las carpetas de manera automática
os.makedirs('src/datos', exist_ok=True)
print("✅ Estructura 'src/datos' creada correctamente.")

✅ Estructura 'src/datos' creada correctamente.


Ahora creamos los archivos de inicialización vacíos necesarios para el módulo de Python.

In [3]:
# Crear archivos __init__.py vacíos
with open('src/__init__.py', 'w') as f:
    pass

with open('src/datos/__init__.py', 'w') as f:
    pass

print("✅ Archivos __init__.py creados con éxito.")

✅ Archivos __init__.py creados con éxito.


Por favor, pega el contenido correspondiente de tus scripts `carga.py`, `preprocesamiento.py`, y `analisis.py` en las celdas de abajo o utiliza el editor integrado de Colab haciendo doble clic sobre los archivos que ahora aparecen en el panel izquierdo dentro de `src/datos/` para guardar tu código de forma definitiva.

In [ ]:
from google.colab import files
import os
import shutil

# Aseguramos que el directorio destino existe
destino = 'src/datos'


print("Selecciona los archivos para la carpeta src/datos/ (ej. carga.py, preprocesamiento.py, analisis.py):")
uploaded = files.upload()

for nombre_archivo, contenido in uploaded.items():
    # Definimos la ruta de destino completa
    ruta_destino = os.path.join(destino, nombre_archivo)

    # Movemos el archivo subido a la carpeta destino
    shutil.move(nombre_archivo, ruta_destino)
    print(f"✅ Archivo '{nombre_archivo}' guardado exitosamente en '{ruta_destino}'")

## Imports

In [4]:
%pip install datasets transformers torch pandas numpy scikit-learn matplotlib

In [2]:
# Actualizamos IPython para corregir la incompatibilidad de autoreload con Python 3.13 TUVE QUE AGREGAR ESTO PARA QUE ME CORRA EL BLOQUE DE ABAJO (Seba)
%pip install --upgrade --quiet ipython

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.18.1 which is incompatible.


In [3]:
# Cargamos la extensión para recarga automática de módulos modificados
%load_ext autoreload
%autoreload 2

# Importamos las funciones de tus scripts subidos
from src.datos.carga import cargar_textos_wikipedia, cargar_opensubtitles
from src.datos.preprocesamiento import construir_dataset, construir_embeddings_por_palabra, generar_matriz_features_rf
from src.datos.analisis import resumen_etiquetas

## Tokenizador

Utilizamos el código proporcionado.

In [4]:
from transformers import BertTokenizer, BertModel
import torch

model_name = "bert-base-multilingual-cased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)
model.eval()

def get_multilingual_token_embedding(token: str):
    """
    Devuelve el embedding (estático) para el token.
    """
    token_id = tokenizer.convert_tokens_to_ids(token)

    if token_id is None or token_id == tokenizer.unk_token_id:
        print(f"❌El token '{token}' no pertenece al vocabulario de multilingual BERT.")
        return None

    embedding_vector = model.embeddings.word_embeddings.weight[token_id]

    #print(f"✅Token: '{token}' | ID: {token_id}")
    #print(f"Embedding shape: {embedding_vector.shape}")
    return embedding_vector

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Dataset

Primero tenemos que conseguir textos para llevar a cabo la tarea. Vamos a utilizar un dataset de Wikipedia (por ahora) después lo podemos cambiar.

MODIFICACIÓN DE LA FUNCIÓN PARA CARGAR TEXTOS: agregué la opción de agregar preguntas en los textos que bajamos. Probé con 300 y funciona bien. Para el entrenamiento entiendo que tenemos que bajar más.

Traté de usar un dataset especial de preguntas o el de opensubtitles y me tiraba un error de HF... Entiendo que con esto está bien igual.

In [5]:
# aleatorio=True toma artículos al azar en vez de los primeros 100 seguidos
textos_wiki, preguntas_wiki = cargar_textos_wikipedia(n_estandar=70, n_preguntas=300, min_len=100, aleatorio=False)

# OpenSubtitles tiene mucha más densidad de preguntas que Wikipedia (~18% de las líneas
# vs. unas pocas cada mil artículos), así que con menos líneas alcanza para la misma cuota.
# min_palabras_pregunta más bajo que en Wikipedia porque en subtítulos las preguntas
# suelen ser cortas ("¿En serio?", "¿Y vos?") y con 4 palabras se pierden muchas.
textos_subs, preguntas_subs = cargar_opensubtitles(n_estandar=70, n_preguntas=300, min_palabras_pregunta=2)

muestra_textos = textos_wiki + preguntas_wiki + textos_subs + preguntas_subs

print(f"Se descargaron {len(muestra_textos)} textos de muestra.")

# Mostramos el primer ejemplo crudo
print("\nEjemplo de texto crudo descargado:")
print(muestra_textos[0][:250] + "...")

Iniciando descarga optimizada en streaming de Wikipedia en español (Versión: 20231101.es)...


README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

✅ Descarga finalizada: 70 textos estándar y 300 oraciones de preguntas obtenidas.
Iniciando descarga en streaming de OpenSubtitles en español...


README.md:   0%|          | 0.00/940 [00:00<?, ?B/s]

✅ Descarga finalizada: 70 líneas estándar y 300 líneas con pregunta obtenidas.
Se descargaron 740 textos de muestra.

Ejemplo de texto crudo descargado:
Anaxyrus debilis es una especie de anfibios de la familia Bufonidae nativo de Estados Unidos, en los estados de Arizona, Nuevo México, Colorado, Kansas y Texas, así como al norte de México, en los estados de Tamaulipas, San Luis Potosí, Durango y Zac...


Una vez que bajamos los parráfos de wikipedia, los vamos a procesar para armar el dataset que utilizaremos luego.

### Funciones de procesamiento

Están en `src/datos/preprocesamiento.py`:
- `normalizar_signos`: reemplaza o elimina los signos que no son etiquetas (`REEMPLAZOS`).
- `seccionar_palabra`: separa una palabra cruda en puntuación inicial, sub-palabras limpias y puntuación final.
- `determinar_capitalizacion`: clase de capitalización (0, 1, 2, 3) según la consigna.
- `procesar_parrafo_a_chunks`: tokeniza con BERT, asigna las etiquetas a cada sub-token y parte en chunks.
- `construir_dataset`: aplica lo anterior a todos los textos y concatena.

Reemplaza la extracción de puntuación que estaba dentro de `procesar_parrafo_a_chunks`. Antes solo se miraba el primer y el último carácter, así que en casos como `"¿Vienes?"` o `(Andorra),` se perdía la etiqueta, y los demás signos quedaban como tokens. Ahora se limpian todos los signos de los bordes, las palabras con guion se separan (`Castilla-La` → `Castilla La`) y se borran los signos internos (`EE.UU` → `EEUU`).

Aplicamos sobre la lista de parráfos bajados.

In [6]:
df_dataset_final = construir_dataset(muestra_textos, tokenizer, chunk_size=32)

print("Muestra del Dataset de Entrenamiento generado por Chunks:")
display(df_dataset_final.head(20))

Muestra del Dataset de Entrenamiento generado por Chunks:


,instancia_id,token_id,token,punt_inicial,punt_final,capitalización
0,1,18812,ana,,,1
1,1,46776,##xy,,,1
2,1,13923,##rus,,,1
3,1,55493,deb,,,0
4,1,34366,##ilis,,,0
5,1,10196,es,,,0
6,1,10153,una,,,0
7,1,15136,especie,,,0
8,1,10104,de,,,0
9,1,10151,an,,,0


Veamos cuantas instacias tenemos

In [7]:
resumen_etiquetas(df_dataset_final)

Número total de tokens (filas en CSV): 71907
Número total de instancias (chunks): 2750

Punt. inicial:
 punt_inicial
(nada)    71314
¿           593
Name: count, dtype: int64 

Punt. final:
 punt_final
(nada)    65734
,          3427
.          2128
?           618
Name: count, dtype: int64 

Capitalización (por palabra):
 capitalización
0    41662
1    10740
2      112
3      480
Name: count, dtype: int64 

Instancias con '?': 21.49%
punt_inicial OK
punt_final OK
capitalización OK


In [8]:
# Verificar si algún chunk comienza con un sub-token
primeros_tokens = df_dataset_final.groupby("instancia_id").first()

chunks_rotos = primeros_tokens[
    primeros_tokens["token"].str.startswith("##")
]

print("Chunks que empiezan con ##:", len(chunks_rotos))

Chunks que empiezan con ##: 0


In [9]:
tamanios = df_dataset_final.groupby("instancia_id").size()

print("Tamaño mínimo:", tamanios.min())
print("Tamaño máximo:", tamanios.max())
print("Chunks con más de 32 tokens:", (tamanios > 32).sum())

Tamaño mínimo: 1
Tamaño máximo: 32
Chunks con más de 32 tokens: 0


## Modelos

### Random Forest (revisar y mejorar)

Utilizamos las funciones ```construir_embeddings_por_palabra```  y ```generar_matriz_features_rf``` para generar los datos de entrenamiento necesarios para entrenar el primer modelo del trabajo.

La idea es entrenar Randoms Forests, donde la entrada será el promedio de los embeddings de cada palabra. Con la primer función hacemos todos los cálculos. Con la segunda agrego la posibilidad de concatenar los embeddings de las palabras vecinas para darle más contexto al algoritmo.

In [10]:
# Esto lo tuve que poner para que no se sature la memoria del colab.
with torch.no_grad():
    df_palabras = construir_embeddings_por_palabra(
        df_dataset_final,
        get_multilingual_token_embedding
    )

print("=== VERIFICACIÓN DEL DATASET PROCESADO ===")
print("Número de palabras encontradas:", len(df_palabras))


print("\nEjemplo del primer registro de palabra:")
print(df_palabras[['instancia_id', 'palabra', 'punt_inicial', 'punt_final', 'capitalización']].head(5))

Procesando embeddings de palabras:   0%|          | 0/52994 [00:00<?, ?it/s]

=== VERIFICACIÓN DEL DATASET PROCESADO ===
Número de palabras encontradas: 52994

Ejemplo del primer registro de palabra:
   instancia_id   palabra punt_inicial punt_final  capitalización
0             1  anaxyrus                                       1
1             1   debilis                                       0
2             1        es                                       0
3             1       una                                       0
4             1   especie                                       0


#### Train / test split

Separamos en train y test por **`instancia_id` entero** (oración completa), no palabra por palabra.

Ojo con la fuga de datos: `generar_matriz_features_rf` arma el `X` de cada palabra concatenando los embeddings de sus vecinas **dentro del mismo `instancia_id`** (ventana de contexto). Si separáramos palabra por palabra al azar, dos palabras vecinas de la misma oración podrían terminar una en train y otra en test, compartiendo casi el mismo `X` porque sus ventanas se superponen — el modelo estaría "viendo" parte del test sin que nos demos cuenta. Por eso separamos primero las oraciones enteras, y recién después armamos las matrices de cada lado por separado. (revisar esto)

In [33]:
from sklearn.model_selection import train_test_split

instancias = df_palabras['instancia_id'].unique()
instancias_train, instancias_test = train_test_split(instancias, test_size=0.2, random_state=23)

df_palabras_train = df_palabras[df_palabras['instancia_id'].isin(instancias_train)]
df_palabras_test = df_palabras[df_palabras['instancia_id'].isin(instancias_test)]

# Armamos las matrices por separado para cada lado: así la ventana de contexto de
# generar_matriz_features_rf nunca mezcla palabras de train con palabras de test.
window_size = 2
X_train, y_init_train, y_fin_train, y_cap_train = generar_matriz_features_rf(df_palabras_train, window_size)
X_test, y_init_test, y_fin_test, y_cap_test = generar_matriz_features_rf(df_palabras_test, window_size)

print(f"Instancias (oraciones): {len(instancias_train)} train / {len(instancias_test)} test")
print(f"Palabras: {len(df_palabras_train)} train / {len(df_palabras_test)} test")
print("Forma de la matriz de entradas X_train:", X_train.shape)
print("Forma de la matriz de entradas X_test:", X_test.shape)
# Por ejemplo, si el embedding es de 768 y window_size=2, la dimensión será (N_palabras, 5 * 768) -> 3840 columnas

Instancias (oraciones): 2200 train / 550 test
Palabras: 42215 train / 10779 test
Forma de la matriz de entradas X_train: (42215, 3840)
Forma de la matriz de entradas X_test: (10779, 3840)


#### Entrenamiento

Usamos un solo `RandomForestClassifier` para las 3 tareas a la vez (`y` de 3 columnas: `punt_inicial`, `punt_final`, `capitalización`), en vez de un RF por tarea (analizar esta posiblidad). Los mismos árboles/splits del bosque sirven para las 3 salidas.

`class_weight='balanced'` porque las clases están muy desbalanceadas (`resumen_etiquetas` mostró 350 `¿` contra 213481 `(nada)`)

In [36]:
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# y_cap_train es numérico (0-3) y y_init_train/y_fin_train son strings; al juntarlas
# en una sola matriz numpy, todo pasa a string (dtype '<U..'). Lo tenemos en cuenta
# al leer las predicciones más abajo.
y_train = np.column_stack([y_init_train, y_fin_train, y_cap_train])

rf = RandomForestClassifier(class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

print(f"Modelo entrenado (con tamaño de ventana = {window_size}). Clases por salida:")
for nombre, clases in zip(['punt_inicial', 'punt_final', 'capitalización'], rf.classes_):
    print(f"  {nombre}: {list(clases)}")

Modelo entrenado (con tamaño de ventana = 2). Clases por salida:
  punt_inicial: [np.str_(''), np.str_('¿')]
  punt_final: [np.str_(''), np.str_(','), np.str_('.'), np.str_('?')]
  capitalización: [np.str_('0'), np.str_('1'), np.str_('2'), np.str_('3')]


#### Evaluación

F1 macro por tarea, con las clases exactas que pide el enunciado para cada una.

In [40]:
from sklearn.metrics import f1_score

y_pred = rf.predict(X_test)
y_init_pred, y_fin_pred, y_cap_pred = y_pred[:, 0], y_pred[:, 1], y_pred[:, 2].astype(int)

f1_init = f1_score(y_init_test, y_init_pred, average='macro', labels=['¿', ''])
f1_fin = f1_score(y_fin_test, y_fin_pred, average='macro', labels=[',', '.', '?', ''])
f1_cap = f1_score(y_cap_test, y_cap_pred, average='macro', labels=[0, 1, 2, 3])

print(f"=== RESULTADOS DE EVALUACIÓN CON WINDOW SIZE = {window_size} ===")
print(f"F1 macro punt_inicial:    {f1_init:.3f}")
print(f"F1 macro punt_final:      {f1_fin:.3f}")
print(f"F1 macro capitalización:  {f1_cap:.3f}")

=== RESULTADOS DE EVALUACIÓN CON WINDOW SIZE = 2 ===
F1 macro punt_inicial:    0.678
F1 macro punt_final:      0.292
F1 macro capitalización:  0.350


Análisis de dónde falla el modelo.

In [43]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

tareas = [
    ("Puntuación inicial", y_init_test, y_init_pred, ["", "¿"]),
    ("Puntuación final", y_fin_test, y_fin_pred, ["", ",", ".", "?"]),
    ("Capitalización", y_cap_test, y_cap_pred, [0, 1, 2, 3])
]

for nombre, y_real, y_estimado, clases in tareas:
    print(f"\n===== {nombre} con window_size = {window_size} =====")

    print("\nReporte por clase:")
    print(classification_report(
        y_real, y_estimado,
        labels=clases,
        zero_division=0
    ))

    print("\nMatriz de confusión:")
    matriz = confusion_matrix(
        y_real, y_estimado, labels=clases
    )

    display(pd.DataFrame(
        matriz,
        index=[f"Real {repr(c)}" for c in clases],
        columns=[f"Predicho {repr(c)}" for c in clases]
    ))


===== Puntuación inicial con window_size = 2 =====

Reporte por clase:
              precision    recall  f1-score   support

                   0.99      1.00      1.00     10668
           ¿       0.89      0.23      0.36       111

    accuracy                           0.99     10779
   macro avg       0.94      0.61      0.68     10779
weighted avg       0.99      0.99      0.99     10779


Matriz de confusión:


,Predicho '',Predicho '¿'
Real '',10665,3
Real '¿',86,25



===== Puntuación final con window_size = 2 =====

Reporte por clase:
              precision    recall  f1-score   support

                   0.88      1.00      0.94      9502
           ,       0.82      0.01      0.03       705
           .       1.00      0.04      0.08       452
           ?       0.89      0.07      0.12       120

    accuracy                           0.88     10779
   macro avg       0.90      0.28      0.29     10779
weighted avg       0.89      0.88      0.83     10779


Matriz de confusión:


,Predicho '',"Predicho ','",Predicho '.',Predicho '?'
Real '',9499,2,0,1
"Real ','",696,9,0,0
Real '.',433,0,19,0
Real '?',112,0,0,8



===== Capitalización con window_size = 2 =====

Reporte por clase:
              precision    recall  f1-score   support

           0       0.82      1.00      0.90      8527
           1       0.93      0.19      0.32      2128
           2       0.00      0.00      0.00        21
           3       1.00      0.10      0.18       103

    accuracy                           0.83     10779
   macro avg       0.69      0.32      0.35     10779
weighted avg       0.84      0.83      0.78     10779


Matriz de confusión:


,Predicho 0,Predicho 1,Predicho 2,Predicho 3
Real 0,8503,24,0,0
Real 1,1717,411,0,0
Real 2,21,0,0,0
Real 3,87,6,0,10


comparar con window_size = 1 y 2?
